Computer Vision

We build tools to solve the problem of binocular vision by working backwards through the videos of Shree Nayar from video 5 on Simple Stereo https://www.youtube.com/watch?v=hUVyDabn1Mg&list=PL2zRqk16wsdoCCLpou-dGo7QQNks1Ppzo&index=5

1. From 2 images solve the correspondence problem, also called stereo matching [find corresponding points] via template matching. Which metric? Which window size? Adaptive?


We begin with our calibration cube, and solve its correspondence problem [matching image points [pixel pairs] to known 3d coordinates]


\begin{equation}\label{eq:world2pix}
  \begin{bmatrix} u^{(n)} \\ v^{(n)} \\ 1 \end{bmatrix} = \begin{bmatrix} p[0,0] & p[0,1] & p[0,2] & p[0,3] \\
                                                        p[1,0] & p[1,1] & p[1,2] & p[1,3] \\
                                                        p[2,0] & p[2,1] & p[2,2] & p[2,3] \end{bmatrix}
                                       \begin{bmatrix} x^{(n)} \\ y^{(n)} \\ z^{(n)} \\ 1 \end{bmatrix}           
\end{equation}

that is

\begin{equation}\label{eq:uvp}
  \eqalign{u^{(n)} &= \frac{p[0,0]x^{(n)} + p[0,1]y^{(n)} + p[0,2]z^{(n)} + p[0,3]}
                           {p[2,0]x^{(n)} + p[2,1]y^{(n)} + p[2,2]z^{(n)} + p[2,3]} \cr
           v^{(n)} &= \frac{p[1,0]x^{(n)} + p[1,1]y^{(n)} + p[1,2]z^{(n)} + p[1,3]}
                           {p[2,0]x^{(n)} + p[2,1]y^{(n)} + p[2,2]z^{(n)} + p[2,3]} \cr}
\end{equation}

that is

\begin{equation}\label{eq:uvp12}
\begin{bmatrix} x^{(n)} & y^{(n)} & z^{(n)} & 1 & 0 & 0 & 0 & 0 & -x^{(n)}u^{(n)} & -y^{(n)}u^{(n)} & -z^{(n)}u^{(n)} & -   u^{(n)} \\
  0 & 0 & 0 & 0 &  x^{(n)} & y^{(n)} & z^{(n)} & 1 & -x^{(n)}v^{(n)} & -y^{(n)}v^{(n)} & -z^{(n)}v^{(n)} & -v^{(n)} \end{bmatrix}
  \begin{bmatrix} p[0,0] \\ p[0,1] \\ p[0,2] \\ p[0,3] \\
                  p[1,0] \\ p[1,1] \\ p[1,2] \\ p[1,3] \\
                  p[2,0] \\ p[2,1] \\ p[2,2] \\ p[2,3] \end{bmatrix}
   = \begin{bmatrix} 0 \\ 0 \end{bmatrix}
\end{equation}

We stack these 27 pairs of equations to arrive at one 54x12 system $Ap=0$ subject to $\|p\|=1$, i.e., $p$ is the eigenvector associated with the least eigenvalue of $A^TA$.

In [3]:
# Calibration Cube   img = plt.imread('../Images/CalCube.jpg')
# used    conda install opencv      from terminal "here"
# Shree Nayar     https://www.youtube.com/watch?v=hUVyDabn1Mg&list=PL2zRqk16wsdoCCLpou-dGo7QQNks1Ppzo&index=5

import cv2
import numpy as np

xpix = np.zeros(27)
ypix = np.zeros(27)
ind = 0

def get_coordinates(event, x, y, flags, param):    # Mouse callback function
    global ind
    if event == cv2.EVENT_LBUTTONDOWN:
        xpix[ind] = x
        ypix[ind] = y
        cv2.putText(imgC, str(ind), (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        ind = ind + 1
        cv2.imshow('image', imgC)

img = cv2.imread('../Images/CalCube.jpg', 0)  # read in grayscale
#img = img_bgr # cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
(m,n) = img.shape
imgC = cv2.resize(img, (int(n/4), int(m/4)))   # downsample

# Create a window and bind the function to it
cv2.namedWindow('image')
cv2.setMouseCallback('image', get_coordinates)

# Main loop to display the image
print('Click on 9 crossings in the (x,z) plane from low to high')
print('then the 9 crossings in the (y,z) plane')
print('then the 9 crossings in the z = 4 plane')
print("Press 'Esc' to exit.")

while True:
    cv2.imshow('image', imgC)
    k = cv2.waitKey(1) & 0xFF
    if k == 27: # 27 is ASCII for the Esc key
        break
        
print(xpix, ypix)

np.savez('xpixANDypix.npz', first=xpix, second=ypix)
        
cv2.destroyAllWindows()

Click on 9 crossings in the (x,z) plane from low to high
then the 9 crossings in the (y,z) plane
then the 9 crossings in the z = 4 plane
Press 'Esc' to exit.
[497. 567. 630. 500. 575. 645. 504. 584. 659. 357. 296. 235. 346. 280.
 217. 340. 266. 199. 421. 510. 593. 343. 430. 511. 269. 353. 437.] [661. 610. 568. 600. 544. 493. 522. 463. 417. 665. 607. 558. 596. 537.
 485. 525. 462. 409. 374. 318. 272. 317. 271. 225. 271. 229. 192.]


In [12]:
# construct the A matrix then the Projection matrix then partition it

import numpy as np
from numpy import linalg as LA

loaded_data = np.load('xpixANDypix.npz')
u = loaded_data['first']   
v = loaded_data['second'] 

#u = xpix
#v = ypix

x = np.array([1, 2, 3, 1, 2, 3, 1, 2, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3, 1, 2, 3, 1, 2, 3])
y = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 1, 1, 2, 2, 2, 3, 3, 3])
z = np.array([1, 1, 1, 2, 2, 2, 3, 3, 3, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4])

A = np.zeros((54,12))

for n in range(27):
    A[2*n,:] = [ x[n], y[n], z[n], 1, 0, 0, 0, 0, -x[n]*u[n], -y[n]*u[n], -z[n]*u[n], -u[n]]
    A[2*n+1,:] = [ 0, 0, 0, 0,  x[n], y[n], z[n], 1, -x[n]*v[n], -y[n]*v[n], -z[n]*v[n], -v[n]]

evals, evecs = LA.eig(A.T@A)

print(evals)

ind = np.argmin(evals)  # index of smallest eigenvalue

p = evecs[ind]   # corresponding eigenvector

print(p)

P = np.reshape(p, (3,4))

print(P)

# now factor

[1.24916103e+08 2.03153088e+07 5.02129757e+06 1.61487687e+06
 3.47105761e+02 5.77851928e+01 1.64080798e+01 1.15343539e+01
 2.37671695e-04 2.29045563e+00 8.29565778e-01 1.47463306e+00]
[-4.79544655e-01  6.89616774e-01 -5.12602765e-01 -1.78048615e-01
 -7.42242245e-04  2.60041494e-04  4.15182598e-04  1.08881042e-05
 -3.95557793e-05 -1.06470600e-03  8.93075050e-04 -3.17274933e-05]
[[-4.79544655e-01  6.89616774e-01 -5.12602765e-01 -1.78048615e-01]
 [-7.42242245e-04  2.60041494e-04  4.15182598e-04  1.08881042e-05]
 [-3.95557793e-05 -1.06470600e-03  8.93075050e-04 -3.17274933e-05]]


Now we recall that

\begin{equation}\label{eq:Pdec}
  P = M_{int}M_{ext} \hskip 0.25in 
   M_{int} = \begin{bmatrix} R & 0 \end{bmatrix},  \hskip 0.25in
   M_{ext} = \begin{bmatrix} Q & t \\ 0 & 1 \end{bmatrix} \hskip 0.25in
   R = \begin{bmatrix} f_x & 0 & o_x\\ 0 & f_y & o_y\\ 0 & 0 & 1 \end{bmatrix}
\end{equation}

from which we deduce

\begin{equation}\label{eq:Pdec1}
  P[:,0:2] = RQ
\end{equation}

produces $R$ and $Q$ through our RQ decompostion. The remaining column of $P$ then determines $t$ via

\begin{equation}\label{eq:Pdec2}
  P[:,3] = Rt
\end{equation}




In [18]:
import numpy as np

def rot(ind, ang):   # build a rotation matrix from axis index (0,1, or 2 for x,y, or z) and angle
    a = np.zeros(3)
    a[ind] = 1
    aO = np.outer(a, a)
    aX = np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])
    K = aO + np.sin(ang)*aX + np.cos(ang)*(np.eye(3)-aO)
    return K

A = P[:,0:3]
if (np.linalg.det(A) < 0):
    A = -A
print("A = ", A)
print(" ")

alpha = np.arctan(-A[2,1]/A[2,2])   # find alpha
Kx = rot(0,alpha)
B = A@Kx

beta = np.arctan(A[2,0]/B[2,2])   # find beta
Ky = rot(1,beta)
C = B@Ky
if (C[2,2] < 0):                  # ensure that C[2,2] > 0
    Ky = rot(1,beta+np.pi)
    C = B@Ky

gamma = np.arctan(-C[1,0]/B[1,1])   # find gamma
Kz = rot(2,gamma)
R = C@Kz
if (R[1,1] < 0):                  # ensure that R[1,1] > 0
    Kz = rot(2,gamma+np.pi)
    R = C@Kz

print("R = ", R)                 # display the factors
print(" ")
Q = Kz.T@Ky.T@Kx.T
print("Q = ", Q)
print(" ")

print("||A-RQ|| = ", np.linalg.norm(A-R@Q))  # display the distance from A to RQ
print(" ")

t = np.linalg.solve(R, P[:,3])

print("t = ", t)

A =  [[ 4.79544655e-01 -6.89616774e-01  5.12602765e-01]
 [ 7.42242245e-04 -2.60041494e-04 -4.15182598e-04]
 [ 3.95557793e-05  1.06470600e-03 -8.93075050e-04]]
 
R =  [[ 2.34030514e-01  4.48938329e-01 -8.43789110e-01]
 [-2.16840434e-19  8.84905721e-04  8.86768561e-05]
 [-6.47747774e-20 -2.27758175e-19  1.39023256e-03]]
 
Q =  [[ 0.5480982   0.52547729  0.65073956]
 [ 0.83592991 -0.37060946 -0.40480837]
 [ 0.02845263  0.76584741 -0.64239256]]
 
||A-RQ|| =  3.140186602624688e-16
 
t =  [-0.87106544  0.01459123 -0.02282172]
